# YZM212 - Makine Öğrenmesi
## III. Laboratuvar: Matris Manipülasyonu, Özdeğerler ve Özvektörler
**Tarih:** 29.03.2026

---

Bu notebook, özdeğer ve özvektör hesaplamalarını sıfırdan (NumPy'ın hazır `eig` fonksiyonu kullanmadan) gerçekleştirmekte ve ardından NumPy `linalg.eig` sonuçları ile karşılaştırmaktadır.

**Referans:** [LucasBN/Eigenvalues-and-Eigenvectors](https://github.com/LucasBN/Eigenvalues-and-Eigenvectors)

## 1. Kütüphaneler

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

print('NumPy version:', np.__version__)

---
## 2. Sıfırdan Özdeğer ve Özvektör Hesaplama

### 2.1 Yöntem: Kuvvet İterasyonu (Power Iteration)

**Kuvvet İterasyonu**, bir matrisin en büyük (dominant) özdeğerini ve ona karşılık gelen özvektörü bulmak için kullanılan iteratif bir yöntemdir.

**Algoritma:**
1. Rastgele bir başlangıç vektörü $\mathbf{b}_0$ seç
2. Her iterasyonda: $\mathbf{b}_{k+1} = \dfrac{A \mathbf{b}_k}{\|A \mathbf{b}_k\|}$  
3. Özvektör $\mathbf{b}_k$'ya yakınsadığında özdeğer: $\lambda = \mathbf{b}_k^T A \mathbf{b}_k$ (Rayleigh Quotient)

**Deflasyon (Deflation):** Dominant özdeğer bulunduktan sonra, matris deflate edilerek bir sonraki özdeğer bulunur:
$$A' = A - \lambda_1 \mathbf{v}_1 \mathbf{v}_1^T$$

In [ ]:
# ===========================================================
# Referans: LucasBN/Eigenvalues-and-Eigenvectors
# Yöntem: Kuvvet İterasyonu + Deflasyon
# ===========================================================

def power_iteration(A, num_iterations=1000, tol=1e-10):
    """
    Kuvvet İterasyonu ile bir matrisin dominant özdeğer
    ve özvektörünü hesaplar.
    
    Parametreler:
        A              : Kare matris (ndarray)
        num_iterations : Maksimum iterasyon sayısı
        tol            : Yakınsama toleransı
    
    Döndürür:
        eigenvalue  : Dominant özdeğer (float)
        eigenvector : Normalize edilmiş özvektör (ndarray)
    """
    n = A.shape[0]
    # Rastgele başlangıç vektörü
    np.random.seed(42)
    b = np.random.rand(n)
    b = b / np.linalg.norm(b)
    
    for _ in range(num_iterations):
        b_new = A @ b
        b_new_norm = np.linalg.norm(b_new)
        if b_new_norm < 1e-15:
            break
        b_new = b_new / b_new_norm
        
        # Yakınsama kontrolü
        if np.linalg.norm(b_new - b) < tol:
            break
        b = b_new
    
    # Rayleigh Quotient ile özdeğeri hesapla
    eigenvalue = (b.T @ A @ b) / (b.T @ b)
    return eigenvalue, b


def deflate(A, eigenvalue, eigenvector):
    """
    Hotelling Deflasyonu: Bulunan özdeğer/özvektör çiftini
    matristen çıkararak bir sonraki dominant özdeğerin
    bulunmasını sağlar.
    
    A' = A - λ * v * v^T
    """
    v = eigenvector.reshape(-1, 1)
    return A - eigenvalue * (v @ v.T)


def compute_eigenvalues_eigenvectors(A, num_iterations=1000, tol=1e-10):
    """
    Tüm özdeğer ve özvektörleri hesaplar.
    Kuvvet İterasyonu + Deflasyon yöntemini kullanır.
    
    Parametreler:
        A              : n×n kare matris
        num_iterations : Maksimum iterasyon sayısı
        tol            : Yakınsama toleransı
    
    Döndürür:
        eigenvalues  : Özdeğerler listesi
        eigenvectors : Özvektörler listesi (sütun bazında)
    """
    n = A.shape[0]
    A_deflated = A.copy().astype(float)
    eigenvalues = []
    eigenvectors = []
    
    for i in range(n):
        lam, v = power_iteration(A_deflated, num_iterations, tol)
        eigenvalues.append(lam)
        eigenvectors.append(v)
        A_deflated = deflate(A_deflated, lam, v)
    
    return np.array(eigenvalues), np.column_stack(eigenvectors)

print('Fonksiyonlar tanımlandı: power_iteration, deflate, compute_eigenvalues_eigenvectors')

---
## 3. Test Matrisi ve Hesaplama

Simetrik bir 3×3 matris kullanıyoruz (gerçek özdeğerleri garanti eder):

In [ ]:
# Test matrisi (simetrik → gerçek özdeğerler)
A = np.array([
    [4, 2, 0],
    [2, 3, 1],
    [0, 1, 2]
], dtype=float)

print('Test Matrisi A:')
print(A)
print(f'\nBoyut: {A.shape}')
print(f'Matris simetrik mi? {np.allclose(A, A.T)}')

In [ ]:
# Sıfırdan hesaplama (Power Iteration + Deflation)
print('=' * 55)
print('  SIFIRDAN HESAPLAMA (Power Iteration + Deflation)')
print('=' * 55)

custom_eigenvalues, custom_eigenvectors = compute_eigenvalues_eigenvectors(A)

print('\nÖzdeğerler (Custom):')
for i, lam in enumerate(custom_eigenvalues):
    print(f'  λ{i+1} = {lam:.8f}')

print('\nÖzvektörler (Custom) [sütun bazında]:')
print(custom_eigenvectors)

---
## 4. NumPy `linalg.eig` ile Hesaplama

In [ ]:
# NumPy eig fonksiyonu ile hesaplama
print('=' * 55)
print('         NumPy linalg.eig ile Hesaplama')
print('=' * 55)

numpy_eigenvalues, numpy_eigenvectors = np.linalg.eig(A)

print('\nÖzdeğerler (NumPy):')
for i, lam in enumerate(numpy_eigenvalues):
    print(f'  λ{i+1} = {lam:.8f}')

print('\nÖzvektörler (NumPy) [sütun bazında]:')
print(numpy_eigenvectors)

---
## 5. Sonuçların Karşılaştırılması

In [ ]:
# --------------------------------------------------
# Her iki yöntemi karşılaştır
# NOT: Özdeğerler farklı sırada olabilir;
# özdeğerlere göre sıralayarak karşılaştırıyoruz.
# --------------------------------------------------

# Sıralama indeksleri
custom_sorted_idx = np.argsort(custom_eigenvalues)[::-1]
numpy_sorted_idx  = np.argsort(numpy_eigenvalues)[::-1]

custom_evals_sorted = custom_eigenvalues[custom_sorted_idx]
numpy_evals_sorted  = numpy_eigenvalues[numpy_sorted_idx].real

custom_evecs_sorted = custom_eigenvectors[:, custom_sorted_idx]
numpy_evecs_sorted  = numpy_eigenvectors[:, numpy_sorted_idx].real

print(f'{"Özdeğer":<6} {"Custom":>16} {"NumPy":>16} {"Fark (abs)":>14}')
print('-' * 55)
for i in range(len(custom_evals_sorted)):
    diff = abs(custom_evals_sorted[i] - numpy_evals_sorted[i])
    print(f'λ{i+1:<5} {custom_evals_sorted[i]:>16.8f} {numpy_evals_sorted[i]:>16.8f} {diff:>14.2e}')

print()
# Doğrulama
eigenvalues_match = np.allclose(custom_evals_sorted, numpy_evals_sorted, atol=1e-5)
print(f'Özdeğerler eşleşiyor mu? {eigenvalues_match}')

# Özvektör doğrulaması: Av = λv
print('\nÖzvektör Doğrulama (A @ v ≈ λ * v):')
for i in range(len(custom_evals_sorted)):
    v = custom_evecs_sorted[:, i]
    lam = custom_evals_sorted[i]
    residual = np.linalg.norm(A @ v - lam * v)
    print(f'  Özvektör {i+1}: hata = {residual:.2e}')

In [ ]:
# --------------------------------------------------
# Görselleştirme
# --------------------------------------------------
fig = plt.figure(figsize=(14, 5))
gs = GridSpec(1, 2, figure=fig)

# --- Sol: Özdeğer Karşılaştırması ---
ax1 = fig.add_subplot(gs[0, 0])
x = np.arange(len(custom_evals_sorted))
width = 0.35

bars1 = ax1.bar(x - width/2, custom_evals_sorted, width,
                label='Custom (Power Iter.)', color='#2E86AB', alpha=0.85)
bars2 = ax1.bar(x + width/2, numpy_evals_sorted, width,
                label='NumPy eig', color='#E84855', alpha=0.85)

ax1.set_xlabel('Özdeğer İndeksi', fontsize=12)
ax1.set_ylabel('Özdeğer Değeri', fontsize=12)
ax1.set_title('Özdeğer Karşılaştırması\n(Custom vs NumPy)', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([f'λ{i+1}' for i in range(len(custom_evals_sorted))])
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.4)

for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# --- Sağ: Özvektörlerin 2D Projeksiyonu ---
ax2 = fig.add_subplot(gs[0, 1])
colors = ['#2E86AB', '#E84855', '#3BB273']
origin = np.zeros(2)

for i in range(custom_evecs_sorted.shape[1]):
    # Custom - boyutu 2D'ye indir (ilk 2 bileşen)
    cv = custom_evecs_sorted[:2, i]
    nv = numpy_evecs_sorted[:2, i]
    # Yön hizalama (işaret belirsizliği)
    if np.dot(cv, nv) < 0:
        nv = -nv
    ax2.quiver(*origin, cv[0], cv[1], angles='xy', scale_units='xy', scale=1,
               color=colors[i], alpha=0.9, width=0.012,
               label=f'Custom v{i+1}')
    ax2.quiver(*origin, nv[0], nv[1], angles='xy', scale_units='xy', scale=1,
               color=colors[i], alpha=0.45, width=0.006, linestyle='dashed',
               label=f'NumPy v{i+1}')

ax2.set_xlim(-1.2, 1.2)
ax2.set_ylim(-1.2, 1.2)
ax2.axhline(0, color='gray', linewidth=0.8)
ax2.axvline(0, color='gray', linewidth=0.8)
ax2.set_xlabel('Bileşen 1', fontsize=12)
ax2.set_ylabel('Bileşen 2', fontsize=12)
ax2.set_title('Özvektörlerin 2D Projeksiyonu\n(İlk 2 bileşen, Koyu=Custom, Açık=NumPy)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=8, loc='lower right')
ax2.grid(alpha=0.3)
ax2.set_aspect('equal')

plt.tight_layout()
plt.savefig('eigenvalue_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: eigenvalue_comparison.png')

---
## 6. PCA Bağlamında Özdeğer/Özvektör Kullanımı

Makine öğrenmesindeki en yaygın kullanım alanı olan **Temel Bileşen Analizi (PCA)**'yı basit bir örnekle gösterelim:

In [ ]:
# --- Basit PCA örneği ---
np.random.seed(0)
N = 100
# Korelasyonlu 2D veri üret
mean = [0, 0]
cov  = [[3, 2], [2, 2]]   # kovaryans matrisi
X = np.random.multivariate_normal(mean, cov, N)

# 1. Ortalama merkezleme
X_centered = X - X.mean(axis=0)

# 2. Kovaryans matrisi
C = (X_centered.T @ X_centered) / (N - 1)
print('Kovaryans Matrisi:')
print(C)

# 3. Kendi yöntemimizle özdeğer/özvektör hesapla
pca_evals_custom, pca_evecs_custom = compute_eigenvalues_eigenvectors(C)

# 4. NumPy ile özdeğer/özvektör hesapla
pca_evals_numpy, pca_evecs_numpy = np.linalg.eig(C)

# Büyükten küçüğe sırala
idx_c = np.argsort(pca_evals_custom)[::-1]
idx_n = np.argsort(pca_evals_numpy)[::-1]
pca_evals_custom = pca_evals_custom[idx_c]
pca_evals_numpy  = pca_evals_numpy[idx_n].real
pca_evecs_custom = pca_evecs_custom[:, idx_c]
pca_evecs_numpy  = pca_evecs_numpy[:, idx_n].real

print(f'\nPCA Özdeğerleri (Custom): {pca_evals_custom.round(4)}')
print(f'PCA Özdeğerleri (NumPy):  {pca_evals_numpy.round(4)}')

# 5. Açıklanan Varyans
explained_var = pca_evals_numpy / pca_evals_numpy.sum() * 100
print(f'\nAçıklanan Varyans (NumPy):')
for i, ev in enumerate(explained_var):
    print(f'  PC{i+1}: %{ev:.2f}')

# Görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Sol: Veri ve temel bileşenler ---
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], alpha=0.5, s=20, color='#5B85AA', label='Veri Noktaları')
scale = 2.5
for i, (ev, color, lbl) in enumerate(zip(pca_evecs_numpy.T,
                                          ['#E84855', '#3BB273'],
                                          ['PC1 (NumPy)', 'PC2 (NumPy)'])):
    ax.quiver(0, 0, ev[0]*scale, ev[1]*scale, angles='xy',
              scale_units='xy', scale=1, color=color, width=0.025, label=lbl)
for i, (ev, color, lbl) in enumerate(zip(pca_evecs_custom.T,
                                          ['#f4a261', '#264653'],
                                          ['PC1 (Custom)', 'PC2 (Custom)'])):
    ax.quiver(0, 0, ev[0]*scale, ev[1]*scale, angles='xy',
              scale_units='xy', scale=1, color=color, width=0.012,
              alpha=0.6, label=lbl)
ax.set_title('PCA: Veriler ve Temel Bileşenler', fontsize=12, fontweight='bold')
ax.set_xlabel('X1'); ax.set_ylabel('X2')
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.3); ax.set_aspect('equal')

# --- Sağ: Açıklanan varyans çubuğu ---
ax2 = axes[1]
bars = ax2.bar(['PC1', 'PC2'], explained_var, color=['#E84855', '#3BB273'], alpha=0.85)
ax2.set_ylim(0, 105)
ax2.set_ylabel('Açıklanan Varyans (%)', fontsize=12)
ax2.set_title('PCA – Açıklanan Varyans', fontsize=12, fontweight='bold')
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'%{bar.get_height():.1f}', ha='center', fontsize=11)
ax2.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('pca_example.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: pca_example.png')

---
## 7. Özet ve Sonuç

In [ ]:
print('=' * 60)
print('                     ÖZET')
print('=' * 60)
print()
print('Kullanılan Matris (A):')
print(A)
print()
print(f'{"":>25} {"λ1":>10} {"λ2":>10} {"λ3":>10}')
print(f'{"Power Iter. (Custom)":>25} '
      + ''.join([f'{v:>10.4f}' for v in custom_evals_sorted]))
print(f'{"NumPy eig":>25} '
      + ''.join([f'{v:>10.4f}' for v in numpy_evals_sorted]))
print()
print('Sonuç: Her iki yöntem de aynı özdeğerleri hesaplamaktadır.')
print('Küçük farklılıklar sayısal hassasiyet ve yakınsama toleransından kaynaklanmaktadır.')